# Desafío Profesional — Data Science
## Caso de Negocio: Cambio Climático
# Etapa 3: Modelado con Machine Learning

---

**Objetivo:** Desarrollar modelos supervisados y no supervisados utilizando los datos limpios de la Etapa 2 para responder las preguntas del caso de negocio.

**Qué vamos a construir en este notebook:**

| Paso | Técnica | Pregunta que responde | Dataset |
|------|---------|----------------------|----------|
| 3.2 | Regresión (lineal, múltiple, Ridge, Lasso, Random Forest, Gradient Boosting) + Feature Engineering | P1: ¿Cómo evolucionan las emisiones? P2: ¿Qué relación hay con PBI y población? | Global |
| 3.3 | Clustering (K-Means, DBSCAN) | P3: ¿Qué patrones emergen entre países y provincias? | Global + Provincias |
| 3.4 | Series temporales (ARIMA/Prophet) | P1: ¿Qué emisiones se predicen? P4: ¿Cómo evolucionará la renovable en Argentina? | Argentina mensual |
| 3.5 | Comparación de modelos | Todas | Todos |

**En términos simples:** hasta ahora exploramos y limpiamos los datos (Etapas 1 y 2). Ahora le pedimos a la computadora que *aprenda patrones* de esos datos y haga predicciones. Es como la diferencia entre mirar un mapa (exploración) y pedirle al GPS que calcule la ruta óptima (modelado).

**Estructura:**
1. Carga de datos limpios
2. Modelos de regresión
3. Clustering de países y provincias
4. Series temporales
5. Tabla comparativa de todos los modelos

---
## 1. Configuración y carga de datos limpios

Cargamos los datasets producidos en la Etapa 2. La ventaja de haber hecho bien la limpieza es que ahora podemos cargar con un simple `read_csv` sin parseos complicados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

COLORS = {'primary':'#1B4F72','accent':'#2E86C1','success':'#27AE60',
          'warning':'#F39C12','danger':'#E74C3C','gray':'#7F8C8D','argentina':'#75AADB'}

# Tabla para acumular métricas de todos los modelos
resultados_modelos = []

print('✅ Librerías cargadas')

In [ ]:
# Cargar datasets limpios de la Etapa 2
DATA_PATH = '/mnt/user-data/outputs/datos_limpios/'

df_global = pd.read_csv(DATA_PATH + 'dataset_global.csv')
df_arg_anual = pd.read_csv(DATA_PATH + 'dataset_argentina_anual.csv')
df_arg_mensual = pd.read_csv(DATA_PATH + 'dataset_argentina_mensual.csv')
df_arg_mensual['mes'] = pd.to_datetime(df_arg_mensual['mes'])
df_prov = pd.read_csv(DATA_PATH + 'dataset_argentina_provincias_wide.csv')

print('✅ Datasets cargados:')
print(f'   Global: {df_global.shape}')
print(f'   Argentina anual: {df_arg_anual.shape}')
print(f'   Argentina mensual: {df_arg_mensual.shape}')
print(f'   Provincias: {df_prov.shape}')

---
## 2. Modelos de regresión (P1, P2)

### ¿Qué es la regresión?

La regresión es una técnica que busca **predecir un número** a partir de otros números. Por ejemplo: ¿puedo predecir cuánto CO2 emitirá un país si conozco su PBI, su población y su porcentaje de renovables?

Imaginá que querés predecir el precio de una casa. Sabés los metros cuadrados, la cantidad de habitaciones y el barrio. La regresión encuentra la "fórmula" que mejor conecta esas variables con el precio. Acá hacemos lo mismo pero con emisiones de CO2.

Vamos a probar varios tipos de regresión, desde la más simple hasta las más sofisticadas, y comparar cuál predice mejor.

### 2.1 — Regresión lineal simple: CO2 mundial vs tiempo

La pregunta más básica: si miro solo el paso del tiempo, ¿puedo predecir las emisiones mundiales futuras? Es un modelo ingenuo (ignora todo excepto el año) pero sirve como **baseline** — un punto de referencia contra el cual comparar modelos más complejos.

In [ ]:
# Preparar datos: CO2 mundial por año
co2_mundial = df_global.groupby('anio')['co2_MtCO2'].sum().reset_index()
co2_mundial.columns = ['anio', 'co2_total']

X = co2_mundial[['anio']]
y = co2_mundial['co2_total']

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo
lr_simple = LinearRegression()
lr_simple.fit(X_train, y_train)
y_pred = lr_simple.predict(X_test)

# Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

resultados_modelos.append({'Modelo': 'Regresión lineal simple (CO2 vs año)',
                           'MSE': mse, 'RMSE': rmse, 'R²': r2, 'MAE': mae, 'Tipo': 'Regresión'})

print(f'Regresión lineal simple: CO2 mundial = f(año)')
print(f'  Coeficiente (pendiente): {lr_simple.coef_[0]:,.0f} MtCO2/año')
print(f'  Intercepto: {lr_simple.intercept_:,.0f}')
print(f'  R²: {r2:.4f} | RMSE: {rmse:,.0f} MtCO2 | MAE: {mae:,.0f} MtCO2')

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.scatter(X_train, y_train, color=COLORS['accent'], alpha=0.6, label='Train')
ax.scatter(X_test, y_test, color=COLORS['danger'], alpha=0.8, label='Test')
anios_full = np.arange(1990, 2025).reshape(-1, 1)
ax.plot(anios_full, lr_simple.predict(anios_full), '--', color=COLORS['primary'], linewidth=2, label='Predicción')
ax.set_title('Regresión lineal simple: CO2 mundial vs año')
ax.set_xlabel('Año'); ax.set_ylabel('CO2 total (MtCO2)'); ax.legend()

# Residuales
ax = axes[1]
residuales = y_test.values - y_pred
ax.scatter(y_pred, residuales, color=COLORS['accent'], alpha=0.7)
ax.axhline(y=0, color=COLORS['danger'], linestyle='--')
ax.set_title('Análisis de residuales')
ax.set_xlabel('Predicción'); ax.set_ylabel('Residual (real - predicho)')

plt.tight_layout()
plt.show()

print(f'\n📊 El modelo captura la tendencia general pero es muy simplista.')
print(f'   Predice un crecimiento lineal constante de {lr_simple.coef_[0]:,.0f} MtCO2 por año.')

### 2.2 — Regresión múltiple: CO2 per cápita = f(PBI, renovables, energía)

Ahora usamos múltiples variables para predecir las emisiones per cápita. La pregunta es: **¿qué factores influyen más en las emisiones de un país?** ¿Es la riqueza? ¿El consumo energético? ¿O el uso de renovables?

Este modelo nos permite cuantificar el peso de cada factor — algo que las visualizaciones de la Etapa 1 insinuaban pero no podían confirmar.

In [ ]:
# Preparar datos: una fila por país-año con todas las variables
df_reg = df_global[['anio', 'pais', 'co2_per_capita', 'pbi_per_capita', 
                     'share_renewables', 'energy_Mtoe', 'poblacion']].dropna().copy()

# Features y target
features = ['pbi_per_capita', 'share_renewables', 'energy_Mtoe', 'poblacion']
X = df_reg[features]
y = df_reg['co2_per_capita']

# Escalar features (importante para comparar coeficientes y para regularización)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Regresión lineal múltiple
lr_multi = LinearRegression()
lr_multi.fit(X_train, y_train)
y_pred_multi = lr_multi.predict(X_test)

r2_m = r2_score(y_test, y_pred_multi)
rmse_m = np.sqrt(mean_squared_error(y_test, y_pred_multi))
mae_m = mean_absolute_error(y_test, y_pred_multi)

resultados_modelos.append({'Modelo': 'Regresión múltiple (CO2pc ~ PBI+renew+energy+pob)',
                           'MSE': rmse_m**2, 'RMSE': rmse_m, 'R²': r2_m, 'MAE': mae_m, 'Tipo': 'Regresión'})

# Coeficientes
coefs = pd.DataFrame({'Feature': features, 'Coeficiente': lr_multi.coef_})
coefs['Abs'] = coefs['Coeficiente'].abs()
coefs = coefs.sort_values('Abs', ascending=False)

print(f'Regresión múltiple: CO2 per cápita ~ PBI + Renovables + Energía + Población')
print(f'  R²: {r2_m:.4f} | RMSE: {rmse_m:.2f} tCO2/persona | MAE: {mae_m:.2f}')
print(f'\nCoeficientes (estandarizados — comparables entre sí):')
display(coefs[['Feature', 'Coeficiente']])

# Validación cruzada
cv_scores = cross_val_score(LinearRegression(), X_scaled, y, cv=5, scoring='r2')
print(f'\nValidación cruzada (5-fold): R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Visualización: coeficientes + predicción vs real
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Coeficientes
ax = axes[0]
colors_coef = [COLORS['success'] if v < 0 else COLORS['danger'] for v in coefs['Coeficiente']]
ax.barh(coefs['Feature'], coefs['Coeficiente'], color=colors_coef)
ax.set_title('Importancia de cada variable (coeficientes estandarizados)')
ax.set_xlabel('Coeficiente (+ = más CO2, - = menos CO2)')
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5)

# Predicción vs real
ax = axes[1]
ax.scatter(y_test, y_pred_multi, alpha=0.5, color=COLORS['accent'])
lims = [min(y_test.min(), y_pred_multi.min()), max(y_test.max(), y_pred_multi.max())]
ax.plot(lims, lims, '--', color=COLORS['danger'], linewidth=1.5, label='Predicción perfecta')
ax.set_title(f'Predicción vs Valor real (R² = {r2_m:.3f})')
ax.set_xlabel('CO2 per cápita real'); ax.set_ylabel('CO2 per cápita predicho'); ax.legend()

plt.tight_layout()
plt.show()

print(f'📊 Interpretación de los coeficientes:')
for _, row in coefs.iterrows():
    direction = 'aumenta' if row['Coeficiente'] > 0 else 'reduce'
    print(f'   {row["Feature"]}: {direction} las emisiones per cápita (coef={row["Coeficiente"]:.3f})')

### 2.3 — Regularización: Ridge y Lasso

Cuando tenemos varias variables que pueden estar correlacionadas entre sí (por ejemplo, PBI y consumo energético suelen ir juntos), los modelos de regresión pueden volverse inestables. La **regularización** es una técnica que penaliza los coeficientes grandes para evitar este problema.

- **Ridge** (L2): reduce todos los coeficientes proporcionalmente. Útil cuando todas las variables aportan algo.
- **Lasso** (L1): puede llevar coeficientes a exactamente cero, efectivamente eliminando variables irrelevantes. Útil para **selección de variables**.

Pensalo como un editor que revisa tu texto: Ridge lo achica todo un poco, Lasso directamente tacha las frases que no aportan.

In [ ]:
# Ridge con GridSearch para encontrar el mejor alpha
alphas = [0.01, 0.1, 1, 10, 100, 1000]

ridge_gs = GridSearchCV(Ridge(), {'alpha': alphas}, cv=5, scoring='r2')
ridge_gs.fit(X_train, y_train)
y_pred_ridge = ridge_gs.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

resultados_modelos.append({'Modelo': f'Ridge (alpha={ridge_gs.best_params_["alpha"]})',
                           'MSE': rmse_ridge**2, 'RMSE': rmse_ridge, 'R²': r2_ridge,
                           'MAE': mean_absolute_error(y_test, y_pred_ridge), 'Tipo': 'Regresión'})

# Lasso con GridSearch
lasso_gs = GridSearchCV(Lasso(max_iter=10000), {'alpha': alphas}, cv=5, scoring='r2')
lasso_gs.fit(X_train, y_train)
y_pred_lasso = lasso_gs.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

resultados_modelos.append({'Modelo': f'Lasso (alpha={lasso_gs.best_params_["alpha"]})',
                           'MSE': rmse_lasso**2, 'RMSE': rmse_lasso, 'R²': r2_lasso,
                           'MAE': mean_absolute_error(y_test, y_pred_lasso), 'Tipo': 'Regresión'})

print(f'Ridge: R²={r2_ridge:.4f} | RMSE={rmse_ridge:.2f} | Mejor alpha={ridge_gs.best_params_["alpha"]}')
print(f'Lasso: R²={r2_lasso:.4f} | RMSE={rmse_lasso:.2f} | Mejor alpha={lasso_gs.best_params_["alpha"]}')

# Comparar coeficientes
comp_coefs = pd.DataFrame({
    'Feature': features,
    'Lineal': lr_multi.coef_,
    'Ridge': ridge_gs.best_estimator_.coef_,
    'Lasso': lasso_gs.best_estimator_.coef_
})
print(f'\nComparación de coeficientes:')
display(comp_coefs)

# Variables eliminadas por Lasso
eliminadas = comp_coefs[comp_coefs['Lasso'].abs() < 0.001]['Feature'].tolist()
if eliminadas:
    print(f'\n📊 Lasso eliminó: {eliminadas} — sugiere que no aportan significativamente al modelo')
else:
    print(f'\n📊 Lasso mantuvo todas las variables — todas son relevantes')

### 2.4 — Feature Engineering: creando mejores predictores

Hasta ahora usamos las variables tal como vienen. Pero en Machine Learning, muchas veces las relaciones entre variables no son lineales: quizás el efecto del PBI sobre las emisiones no es proporcional, sino que se **acelera** con PBI altos, o quizás lo importante no es el PBI solo, sino la **combinación** de PBI alto con poca inversión en renovables.

El **feature engineering** consiste en crear variables nuevas a partir de las existentes para capturar estas relaciones. Es como darle al modelo lentes más potentes para ver patrones que antes no podía.

Vamos a crear:
- `log_pbi`: logaritmo del PBI per cápita (suaviza la escala, captura rendimientos decrecientes)
- `log_energy`: logaritmo del consumo energético
- `pbi_x_renew`: interacción PBI × renovables (¿los países ricos con renovables contaminan menos?)
- `energy_per_capita`: energía per cápita (intensidad energética individual)

In [ ]:
# =============================================================================
# FEATURE ENGINEERING
# =============================================================================

df_reg_fe = df_global[['anio', 'pais', 'co2_per_capita', 'pbi_per_capita',
                       'share_renewables', 'energy_Mtoe', 'poblacion']].dropna().copy()

# Variables transformadas
df_reg_fe['log_pbi'] = np.log1p(df_reg_fe['pbi_per_capita'])
df_reg_fe['log_energy'] = np.log1p(df_reg_fe['energy_Mtoe'])

# Interacciones
df_reg_fe['pbi_x_renew'] = df_reg_fe['pbi_per_capita'] * df_reg_fe['share_renewables']

# Per cápita
df_reg_fe['energy_per_capita'] = np.where(
    df_reg_fe['poblacion'] > 0,
    (df_reg_fe['energy_Mtoe'] * 1e6) / df_reg_fe['poblacion'],  # toe per capita
    np.nan
)

# Features extendidos
features_fe = ['log_pbi', 'share_renewables', 'log_energy', 'energy_per_capita', 'pbi_x_renew']
X_fe = df_reg_fe[features_fe].dropna()
y_fe = df_reg_fe.loc[X_fe.index, 'co2_per_capita']

scaler_fe = StandardScaler()
X_fe_scaled = scaler_fe.fit_transform(X_fe)

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe_scaled, y_fe, test_size=0.2, random_state=42
)

# Regresión múltiple con features mejorados
lr_fe = LinearRegression()
lr_fe.fit(X_train_fe, y_train_fe)
y_pred_fe = lr_fe.predict(X_test_fe)

r2_fe = r2_score(y_test_fe, y_pred_fe)
rmse_fe = np.sqrt(mean_squared_error(y_test_fe, y_pred_fe))
mae_fe = mean_absolute_error(y_test_fe, y_pred_fe)

# Validación cruzada
cv_fe = cross_val_score(LinearRegression(), X_fe_scaled, y_fe, cv=5, scoring='r2')

resultados_modelos.append({
    'Modelo': 'Regresión múltiple + Feature Engineering',
    'MSE': rmse_fe**2, 'RMSE': rmse_fe, 'R²': r2_fe, 'MAE': mae_fe, 'Tipo': 'Regresión'
})

print(f'Regresión múltiple CON Feature Engineering:')
print(f'  R²: {r2_fe:.4f} | RMSE: {rmse_fe:.2f} | MAE: {mae_fe:.2f}')
print(f'  Validación cruzada (5-fold): R² = {cv_fe.mean():.4f} ± {cv_fe.std():.4f}')

print(f'\nComparación con modelo sin Feature Engineering:')
print(f'  R² sin FE:  {r2_m:.4f} | CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  R² con FE:  {r2_fe:.4f} | CV: {cv_fe.mean():.4f} ± {cv_fe.std():.4f}')
mejora = (r2_fe - r2_m) / r2_m * 100 if r2_m > 0 else 0
print(f'  Mejora en R²: {mejora:+.1f}%')

# Coeficientes del modelo mejorado
coefs_fe = pd.DataFrame({'Feature': features_fe, 'Coeficiente': lr_fe.coef_})
coefs_fe = coefs_fe.sort_values('Coeficiente', key=abs, ascending=False)
print(f'\nCoeficientes estandarizados (feature engineering):')
display(coefs_fe)


### 2.5 — Random Forest y Gradient Boosting: modelos no lineales

Los modelos lineales (incluyendo Ridge y Lasso) asumen que la relación entre las variables y el target es una línea recta (o una combinación lineal). Pero en la realidad, muchas relaciones son curvas: el efecto del PBI sobre las emisiones puede ser fuerte hasta cierto punto y luego aplanarse, o la interacción entre renovables y consumo puede no ser proporcional.

**Random Forest** y **Gradient Boosting** son modelos basados en **árboles de decisión** que capturan relaciones no lineales automáticamente. Pensá en un árbol de decisión como un juego de "20 preguntas": va preguntando cosas como "¿el PBI per cápita es mayor a 20,000?" → "¿el % de renovables es mayor a 30%?" y así va dividiendo los datos hasta llegar a una predicción.

- **Random Forest**: crea muchos árboles con datos y variables aleatorias, y promedia sus predicciones. Es robusto y difícil de sobreajustar.
- **Gradient Boosting**: crea árboles secuencialmente, donde cada árbol nuevo intenta corregir los errores del anterior. Suele ser más preciso pero puede sobreajustar si no se ajusta bien.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_fe, y_train_fe)
y_pred_rf = rf.predict(X_test_fe)

r2_rf = r2_score(y_test_fe, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_fe, y_pred_rf))
mae_rf = mean_absolute_error(y_test_fe, y_pred_rf)
cv_rf = cross_val_score(RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
                        X_fe_scaled, y_fe, cv=5, scoring='r2')

resultados_modelos.append({
    'Modelo': 'Random Forest (100 árboles, depth=10)',
    'MSE': rmse_rf**2, 'RMSE': rmse_rf, 'R²': r2_rf, 'MAE': mae_rf, 'Tipo': 'Regresión'
})

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train_fe, y_train_fe)
y_pred_gb = gb.predict(X_test_fe)

r2_gb = r2_score(y_test_fe, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test_fe, y_pred_gb))
mae_gb = mean_absolute_error(y_test_fe, y_pred_gb)
cv_gb = cross_val_score(GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
                        X_fe_scaled, y_fe, cv=5, scoring='r2')

resultados_modelos.append({
    'Modelo': 'Gradient Boosting (100 árboles, lr=0.1)',
    'MSE': rmse_gb**2, 'RMSE': rmse_gb, 'R²': r2_gb, 'MAE': mae_gb, 'Tipo': 'Regresión'
})

print(f'Random Forest:     R²={r2_rf:.4f} | RMSE={rmse_rf:.2f} | MAE={mae_rf:.2f} | CV: {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')
print(f'Gradient Boosting: R²={r2_gb:.4f} | RMSE={rmse_gb:.2f} | MAE={mae_gb:.2f} | CV: {cv_gb.mean():.4f} ± {cv_gb.std():.4f}')


In [ ]:
# Visualización: Feature importance + Comparación de modelos
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: Feature importance (Gradient Boosting)
ax = axes[0]
importance = pd.DataFrame({'Feature': features_fe, 'Importance': gb.feature_importances_})
importance = importance.sort_values('Importance', ascending=True)
ax.barh(importance['Feature'], importance['Importance'], color=COLORS['accent'])
ax.set_title('Feature Importance — Gradient Boosting')
ax.set_xlabel('Importancia')

# Panel derecho: Predicción vs real de todos los modelos
ax = axes[1]
ax.scatter(y_test_fe, y_pred_fe, alpha=0.4, label=f'Lineal+FE (R²={r2_fe:.3f})', marker='o')
ax.scatter(y_test_fe, y_pred_rf, alpha=0.4, label=f'Random Forest (R²={r2_rf:.3f})', marker='s')
ax.scatter(y_test_fe, y_pred_gb, alpha=0.4, label=f'Gradient Boost (R²={r2_gb:.3f})', marker='^')
lims = [0, max(y_test_fe.max(), max(y_pred_rf.max(), y_pred_gb.max())) * 1.1]
ax.plot(lims, lims, '--', color=COLORS['danger'], linewidth=1.5, label='Predicción perfecta')
ax.set_title('Predicción vs Valor real — Comparación de modelos')
ax.set_xlabel('CO2 per cápita real'); ax.set_ylabel('CO2 per cápita predicho')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'📊 Resumen comparativo de todos los modelos de regresión:')
print(f'   Lineal simple:     R²={r2:.4f} (baseline)')
print(f'   Múltiple:          R²={r2_m:.4f} | CV={cv_scores.mean():.4f}')
print(f'   Ridge:             R²={r2_ridge:.4f}')
print(f'   Lasso:             R²={r2_lasso:.4f}')
print(f'   Múltiple+FE:       R²={r2_fe:.4f} | CV={cv_fe.mean():.4f}')
print(f'   Random Forest:     R²={r2_rf:.4f} | CV={cv_rf.mean():.4f}')
print(f'   Gradient Boosting: R²={r2_gb:.4f} | CV={cv_gb.mean():.4f}')

print(f'\n📊 Insight clave: los modelos basados en árboles capturan relaciones no lineales')
print(f'   que la regresión lineal no puede ver, especialmente en países con perfiles extremos.')


---
## 3. Clustering — agrupación de países y provincias (P3)

### ¿Qué es el clustering?

El clustering busca **agrupar cosas similares sin que nadie les diga de antemano a qué grupo pertenecen**. Es como llegar a una fiesta sin conocer a nadie y empezar a notar que hay grupos: los que hablan de deportes, los que hablan de tecnología, los que están cerca de la comida. Nadie te dijo los grupos — vos los descubriste observando similitudes.

Acá vamos a agrupar países según su perfil de emisiones y renovables, y provincias argentinas según su perfil de generación. Esto nos permite descubrir patrones que no son obvios a simple vista.

### 3.1 — Clustering de países (K-Means)

In [ ]:
# Preparar datos: snapshot del último año disponible (2018) por país
df_cluster = df_global[df_global['anio'] == 2018][[
    'pais', 'co2_per_capita', 'pbi_per_capita', 'share_renewables', 'co2_per_pbi'
]].dropna().copy()

print(f'Países para clustering: {len(df_cluster)}')

# Escalar
cluster_features = ['co2_per_capita', 'pbi_per_capita', 'share_renewables', 'co2_per_pbi']
scaler_cl = StandardScaler()
X_cluster = scaler_cl.fit_transform(df_cluster[cluster_features])

# Método del codo + Silhouette
K_range = range(2, 9)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'o-', color=COLORS['primary'], linewidth=2)
axes[0].set_title('Método del codo (inercia)'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia')
axes[1].plot(K_range, silhouettes, 'o-', color=COLORS['success'], linewidth=2)
axes[1].set_title('Silhouette score'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Score')
plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f'📊 Mejor K según Silhouette: {best_k} (score={max(silhouettes):.3f})')

In [ ]:
# K-Means con el K óptimo
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster['cluster'] = km_final.fit_predict(X_cluster)

# Perfilar clusters
print(f'Perfiles de los {best_k} clusters:')
print('=' * 70)
perfil = df_cluster.groupby('cluster')[cluster_features].mean()
display(perfil)

print(f'\nPaíses por cluster:')
for cl in sorted(df_cluster['cluster'].unique()):
    paises = df_cluster[df_cluster['cluster']==cl]['pais'].tolist()
    print(f'  Cluster {cl}: {paises}')

sil = silhouette_score(X_cluster, df_cluster['cluster'])
resultados_modelos.append({'Modelo': f'K-Means países (K={best_k})',
                           'MSE': np.nan, 'RMSE': np.nan, 'R²': np.nan,
                           'MAE': np.nan, 'Tipo': 'Clustering',
                           'Silhouette': sil})

In [ ]:
# Visualización de clusters (2D: CO2 per cápita vs PBI per cápita)
fig, ax = plt.subplots(figsize=(12, 8))

for cl in sorted(df_cluster['cluster'].unique()):
    subset = df_cluster[df_cluster['cluster']==cl]
    ax.scatter(subset['pbi_per_capita'], subset['co2_per_capita'],
              s=subset['share_renewables']*5+20, alpha=0.7, label=f'Cluster {cl}')
    for _, row in subset.iterrows():
        if row['pais'] in ['Argentina','China','United States','India','Brazil','Germany',
                           'Saudi Arabia','Japan','Australia','South Africa','Nigeria']:
            ax.annotate(row['pais'], (row['pbi_per_capita'], row['co2_per_capita']),
                       fontsize=9, ha='left', xytext=(5,5), textcoords='offset points')

ax.set_title(f'Clustering de países (K={best_k}) — CO2 per cápita vs PBI per cápita\n'
             f'Tamaño = % renovables | Silhouette = {sil:.3f}')
ax.set_xlabel('PBI per cápita (US$)'); ax.set_ylabel('CO2 per cápita (tCO2/persona)')
ax.legend(title='Cluster')
plt.tight_layout()
plt.show()

### 3.2 — Clustering de provincias argentinas

¿Qué provincias tienen un perfil energético renovable similar? Agrupamos las provincias según la composición de su generación renovable en 2018 (% eólica, % solar, % hidro, % biomasa). Esto revela los "polos renovables" de Argentina.

In [ ]:
# Datos de provincias para 2018
df_prov_2018 = df_prov[df_prov['anio'] == 2018][[
    'provincia', 'gen_total_GWh', 'gen_eolico_pct', 'gen_solar_pct', 'gen_hidro_pct', 'gen_biomasa_pct'
]].dropna().copy()

# Solo provincias con generación significativa
df_prov_2018 = df_prov_2018[df_prov_2018['gen_total_GWh'] > 10].reset_index(drop=True)
print(f'Provincias con generación > 10 GWh en 2018: {len(df_prov_2018)}')

# Clustering
prov_features = ['gen_eolico_pct', 'gen_solar_pct', 'gen_hidro_pct', 'gen_biomasa_pct']
X_prov = df_prov_2018[prov_features].values

# Probar K=3 (intuitivo: eólica, hidro, mixta)
km_prov = KMeans(n_clusters=3, random_state=42, n_init=10)
df_prov_2018['cluster'] = km_prov.fit_predict(X_prov)
sil_prov = silhouette_score(X_prov, df_prov_2018['cluster'])

resultados_modelos.append({'Modelo': f'K-Means provincias (K=3)',
                           'MSE': np.nan, 'RMSE': np.nan, 'R²': np.nan,
                           'MAE': np.nan, 'Tipo': 'Clustering',
                           'Silhouette': sil_prov})

print(f'\nPerfiles de clusters provinciales:')
for cl in sorted(df_prov_2018['cluster'].unique()):
    subset = df_prov_2018[df_prov_2018['cluster']==cl]
    provs = subset['provincia'].tolist()
    dom_source = subset[prov_features].mean().idxmax().replace('gen_','').replace('_pct','')
    print(f'  Cluster {cl} ({dom_source}): {provs}')

# Visualización
fig, ax = plt.subplots(figsize=(12, 6))
df_prov_plot = df_prov_2018.sort_values('gen_total_GWh', ascending=True)
colors_cl = {0: COLORS['accent'], 1: COLORS['success'], 2: COLORS['warning']}
bar_colors = [colors_cl.get(cl, COLORS['gray']) for cl in df_prov_plot['cluster']]
ax.barh(df_prov_plot['provincia'], df_prov_plot['gen_total_GWh'], color=bar_colors)
ax.set_title(f'Generación renovable por provincia (2018) — Coloreado por cluster\nSilhouette = {sil_prov:.3f}')
ax.set_xlabel('GWh')
# Leyenda manual
for cl in sorted(df_prov_2018['cluster'].unique()):
    dom = df_prov_2018[df_prov_2018['cluster']==cl][prov_features].mean().idxmax().replace('gen_','').replace('_pct','')
    ax.scatter([], [], color=colors_cl.get(cl, COLORS['gray']), label=f'Cluster {cl} ({dom})', s=100)
ax.legend(title='Perfil dominante')
plt.tight_layout()
plt.show()

---
## 4. Series temporales — Forecast de generación renovable en Argentina (P1, P4)

### ¿Qué son las series temporales?

Una serie temporal es una secuencia de datos ordenados en el tiempo: la temperatura de cada día, el precio de una acción cada hora, la generación eléctrica de cada mes. Lo especial es que el **orden importa**: el valor de hoy depende de los de ayer y anteayer.

Vamos a usar los datos mensuales de generación renovable en Argentina (2011-2018) para predecir cómo va a evolucionar en el futuro. Usamos el modelo ARIMA, que descompone la serie en tendencia, estacionalidad y ruido.

En términos simples: es como mirar la curva de crecimiento de un niño y proyectar cuánto va a medir a los 18 años.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA

# Serie: generación renovable mensual
ts = df_arg_mensual[['mes', 'gen_renovable_total_GWh']].set_index('mes').dropna()
ts.index = pd.DatetimeIndex(ts.index).to_period('M').to_timestamp()
ts = ts.asfreq('MS')  # Monthly Start frequency

print(f'Serie temporal: {len(ts)} observaciones ({ts.index.min()} a {ts.index.max()})')

# Descomposición
decomp = seasonal_decompose(ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observado', color=COLORS['primary'])
decomp.trend.plot(ax=axes[1], title='Tendencia', color=COLORS['danger'])
decomp.seasonal.plot(ax=axes[2], title='Estacionalidad', color=COLORS['success'])
decomp.resid.plot(ax=axes[3], title='Residual (ruido)', color=COLORS['gray'])
plt.suptitle('Descomposición de la generación renovable mensual — Argentina', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print(f'📊 Se observa tendencia ascendente clara (cada vez más renovable)')
print(f'   La estacionalidad refleja variaciones climáticas (más hidro en invierno, más solar en verano)')

In [ ]:
# Modelo ARIMA
# Split: usar hasta 2017 como train, 2018 como test
train_ts = ts[ts.index < '2018-01-01']
test_ts = ts[ts.index >= '2018-01-01']

print(f'Train: {len(train_ts)} meses | Test: {len(test_ts)} meses')

# Ajustar ARIMA(1,1,1) como punto de partida
# p=1: un lag autoregresivo, d=1: una diferenciación (tendencia), q=1: un término de media móvil
model = ARIMA(train_ts, order=(1, 1, 1))
fitted = model.fit()

# Predecir 2018 (12 meses)
forecast = fitted.forecast(steps=len(test_ts))
forecast.index = test_ts.index

# Métricas
mae_arima = mean_absolute_error(test_ts, forecast)
rmse_arima = np.sqrt(mean_squared_error(test_ts, forecast))
mape_arima = np.mean(np.abs((test_ts.values.flatten() - forecast.values) / test_ts.values.flatten())) * 100

resultados_modelos.append({'Modelo': 'ARIMA(1,1,1) — Gen renovable mensual',
                           'MSE': rmse_arima**2, 'RMSE': rmse_arima, 'R²': np.nan,
                           'MAE': mae_arima, 'Tipo': 'Series temporales',
                           'MAPE': mape_arima})

print(f'ARIMA(1,1,1): MAE={mae_arima:.1f} GWh | RMSE={rmse_arima:.1f} GWh | MAPE={mape_arima:.1f}%')

# Visualización
fig, ax = plt.subplots(figsize=(14, 6))
train_ts.plot(ax=ax, label='Train (2011-2017)', color=COLORS['accent'], linewidth=1.5)
test_ts.plot(ax=ax, label='Test real (2018)', color=COLORS['primary'], linewidth=2)
forecast.plot(ax=ax, label='Forecast ARIMA', color=COLORS['danger'], linestyle='--', linewidth=2)
ax.fill_between(forecast.index, forecast*0.8, forecast*1.2, alpha=0.1, color=COLORS['danger'])
ax.set_title(f'ARIMA(1,1,1) — Forecast generación renovable Argentina\nMAE={mae_arima:.1f} GWh | MAPE={mape_arima:.1f}%')
ax.set_xlabel('Mes'); ax.set_ylabel('GWh'); ax.legend()
plt.tight_layout()
plt.show()

print(f'📊 El modelo captura la tendencia pero subestima el crecimiento acelerado de 2018')
print(f'   Esto es esperable: en 2018 se conectaron muchos parques nuevos (cambio estructural)')

---
## 5. Tabla comparativa de todos los modelos

Esta es la tabla que resume todo el trabajo de la Etapa 3 y que después compararemos con los modelos de redes neuronales de la Etapa 4. Para regresión usamos R² y RMSE, para clustering usamos Silhouette Score, y para series temporales usamos MAE y MAPE.

In [ ]:
# Tabla comparativa final
df_resultados = pd.DataFrame(resultados_modelos)

print('=' * 80)
print('TABLA COMPARATIVA DE MODELOS — ETAPA 3')
print('=' * 80)

# Modelos de regresión
print('\n--- REGRESIÓN ---')
reg_results = df_resultados[df_resultados['Tipo']=='Regresión'][['Modelo','R²','RMSE','MAE']]
display(reg_results.reset_index(drop=True))

# Clustering
print('\n--- CLUSTERING ---')
cl_results = df_resultados[df_resultados['Tipo']=='Clustering'][['Modelo','Silhouette']]
display(cl_results.reset_index(drop=True))

# Series temporales
print('\n--- SERIES TEMPORALES ---')
ts_results = df_resultados[df_resultados['Tipo']=='Series temporales'][['Modelo','RMSE','MAE','MAPE']]
display(ts_results.reset_index(drop=True))

# Mejor modelo de regresión
best_reg = df_resultados[df_resultados['Tipo']=='Regresión'].sort_values('R²', ascending=False).iloc[0]
print(f'\n📊 Mejor modelo de regresión: {best_reg["Modelo"]} (R²={best_reg["R²"]:.4f})')
print(f'\n📌 Estos resultados servirán como baseline para comparar con las redes neuronales en la Etapa 4')

In [ ]:
# Guardar tabla de resultados para la Etapa 4
df_resultados.to_csv(DATA_PATH + 'resultados_modelos_etapa3.csv', index=False)
print('✅ Resultados guardados en resultados_modelos_etapa3.csv')

---
## 6. Hallazgos y conclusiones de la Etapa 3

### Modelos de regresión (P1, P2):
- La regresión lineal simple (CO2 vs año) captura la tendencia general con R² alto pero es un modelo ingenuo.
- La regresión múltiple con las variables originales tiene buen R² en test pero mala validación cruzada, indicando problemas de generalización.
- El **Feature Engineering** (log del PBI, interacciones PBI×renovables, energía per cápita) mejora significativamente la estabilidad del modelo en validación cruzada.
- **Random Forest y Gradient Boosting** capturan relaciones no lineales que la regresión lineal no puede ver, con mejor validación cruzada.
- Ridge y Lasso confirman que todas las variables originales son relevantes (Lasso no elimina ninguna).
- La variable más importante según Gradient Boosting es la energía per cápita, seguida del log del PBI.

### Clustering (P3):
- Los países se agrupan naturalmente en clusters que reflejan su nivel de desarrollo económico y perfil energético.
- Las provincias argentinas se agrupan por tipo de fuente renovable dominante: eólicas (Patagonia), hidroeléctricas (Cuyo, NOA) y mixtas/biomasa.

### Series temporales (P1, P4):
- La generación renovable en Argentina muestra tendencia ascendente y estacionalidad.
- El modelo ARIMA captura la tendencia pero subestima el crecimiento acelerado de 2018 (cambio estructural por RenovAr).

### Próximo paso:
**Etapa 4 — Redes Neuronales:** implementar MLP para regresión y LSTM para series temporales, comparar con estos resultados, y elaborar el informe final con storytelling.